# NSRAG Phase 1 — Baseline Evaluation
**Runtime:** Colab T4 GPU (`Runtime → Change runtime type → T4 GPU`)

**Add your OpenAI key:** Colab left sidebar 🔑 Secrets → `OPENAI_API_KEY`

Run cells in order: 1 → 8.

In [ ]:
# ── CELL 1: Mount Drive + setup ─────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT = "/content/drive/MyDrive/NSRAG"
os.makedirs(f"{PROJECT}/results/flashrag", exist_ok=True)
os.makedirs(f"{PROJECT}/data/hotpotqa", exist_ok=True)
os.makedirs(f"{PROJECT}/data/musique", exist_ok=True)
os.makedirs(f"{PROJECT}/data/2wikimultihop", exist_ok=True)
os.makedirs(f"{PROJECT}/data/theoremqa", exist_ok=True)
os.chdir(PROJECT)
print("Working dir:", os.getcwd())

In [ ]:
# ── CELL 2: Install deps on Colab ───────────────────────────────────
%%capture
!pip install transformers sentence-transformers faiss-gpu datasets evaluate
!pip install langchain langchain-community openai anthropic
!pip install nltk rouge-score tqdm rich sympy z3-solver

# Verify GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── CELL 3: Download all datasets ───────────────────────────────────
from datasets import load_dataset
import json, random, os

random.seed(42)

def save_sample(dataset, split, path, n=1000):
    data = list(dataset[split])
    sample = random.sample(data, min(n, len(data)))
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        json.dump(sample, f)
    print(f"  Saved {len(sample)} → {path}")
    return sample

print("HotpotQA...")
hp = load_dataset("hotpot_qa", "fullwiki", trust_remote_code=True)
hp_dev = save_sample(hp, "validation", "data/hotpotqa/dev_1k.json")

print("MuSiQue...")
try:
    mq = load_dataset("musique_qa", trust_remote_code=True)
    save_sample(mq, "validation", "data/musique/dev_1k.json")
except Exception as e:
    print(f"  Primary failed ({e}), trying mirror...")
    mq = load_dataset("alakahanar/musique", trust_remote_code=True)
    save_sample(mq, "validation", "data/musique/dev_1k.json")

print("2WikiMultiHopQA...")
w2 = load_dataset("voidful/2wikimultihopqa", trust_remote_code=True)
save_sample(w2, "validation", "data/2wikimultihop/dev_1k.json")

print("TheoremQA...")
tq = load_dataset("TIGER-Lab/TheoremQA", trust_remote_code=True)
with open("data/theoremqa/test.json", "w") as f:
    json.dump(list(tq["test"]), f)
print(f"  TheoremQA: {len(tq['test'])} samples")

print("\nDone.")

In [ ]:
# ── CELL 4: Eval harness ────────────────────────────────────────────
import re, string
from collections import Counter
from typing import List, Dict

def normalize_answer(s):
    def remove_articles(text): return re.sub(r'\b(a|an|the)\b', ' ', text)
    def white_space_fix(text): return ' '.join(text.split())
    def remove_punc(text):
        exclude = set(string.punctuation)
        return ''.join(ch for ch in text if ch not in exclude)
    return white_space_fix(remove_articles(remove_punc(s.lower())))

def exact_match(pred, golds):
    return float(any(normalize_answer(pred) == normalize_answer(g) for g in golds))

def f1_score(pred, golds):
    def token_f1(p, g):
        pt = normalize_answer(p).split()
        gt = normalize_answer(g).split()
        common = Counter(pt) & Counter(gt)
        n = sum(common.values())
        if n == 0: return 0.0
        prec = n / len(pt)
        rec  = n / len(gt)
        return 2 * prec * rec / (prec + rec)
    return max(token_f1(pred, g) for g in golds)

def evaluate(predictions):
    em_scores, f1_scores, hop_data = [], [], {}
    for item in predictions:
        pred  = item.get("prediction", "")
        golds = item.get("gold_answers", [])
        if isinstance(golds, str): golds = [golds]
        em = exact_match(pred, golds)
        f1 = f1_score(pred, golds)
        em_scores.append(em)
        f1_scores.append(f1)
        h = item.get("hops", -1)
        hop_data.setdefault(h, {"em": [], "f1": []})
        hop_data[h]["em"].append(em)
        hop_data[h]["f1"].append(f1)

    n = len(predictions)
    return {
        "n": n,
        "EM": round(100 * sum(em_scores) / n, 2),
        "F1": round(100 * sum(f1_scores) / n, 2),
        "by_hop": {
            str(k): {
                "EM": round(100*sum(v["em"])/len(v["em"]), 2),
                "F1": round(100*sum(v["f1"])/len(v["f1"]), 2),
                "n":  len(v["em"])
            } for k, v in sorted(hop_data.items())
        }
    }

# Sanity check — expected: EM=33.33, F1=44.44
test = [
    {"prediction": "Paris",          "gold_answers": ["Paris"],         "hops": 1},
    {"prediction": "london",          "gold_answers": ["Paris"],         "hops": 2},
    {"prediction": "The Eiffel Tower", "gold_answers": ["eiffel tower"], "hops": 2},
]
r = evaluate(test)
print(f"Eval harness check → EM: {r['EM']}  F1: {r['F1']}")
assert r["EM"] == 33.33, f"ERROR: expected EM=33.33, got {r['EM']}"
print("PASS")

In [ ]:
# ── CELL 5: Build embedding index ──────────────────────────────────
from sentence_transformers import SentenceTransformer
import faiss, numpy as np, time, json

print("Loading embedder (bge-large-en-v1.5)...")
embedder = SentenceTransformer("BAAI/bge-large-en-v1.5", device="cuda")
print("Embedder loaded on GPU")

def build_corpus(data_path):
    """Extract unique passages from HotpotQA context field."""
    with open(data_path) as f:
        data = json.load(f)
    docs = {}
    for item in data:
        ctx = item.get("context", {})
        titles    = ctx.get("title", [])
        sentences = ctx.get("sentences", [])
        for title, sents in zip(titles, sentences):
            text = " ".join(sents)
            if title not in docs:
                docs[title] = {"title": title, "text": text}
    corpus = list(docs.values())
    print(f"Corpus: {len(corpus)} unique passages")
    return corpus

def build_index(corpus, batch_size=256):
    texts = [f"{d['title']} {d['text']}" for d in corpus]
    t0 = time.time()
    print(f"Encoding {len(texts)} passages...")
    embs = embedder.encode(
        texts, batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True,
        device="cuda"
    )
    print(f"Encoded in {time.time()-t0:.1f}s")
    dim = embs.shape[1]
    res = faiss.StandardGpuResources()
    flat = faiss.IndexFlatIP(dim)
    index = faiss.index_cpu_to_gpu(res, 0, flat)
    index.add(embs.astype(np.float32))
    print(f"FAISS index: {index.ntotal} vectors, dim={dim}")
    return index, embs

corpus_hp = build_corpus("data/hotpotqa/dev_1k.json")
index_hp, embs_hp = build_index(corpus_hp)
print("Index ready.")

In [ ]:
# ── CELL 6: Naive RAG run ───────────────────────────────────────────
import openai, time
from tqdm import tqdm

# ← paste your NEW key here (session only, never saved to Drive)
os.environ["OPENAI_API_KEY"] = "sk-proj-REPLACE_ME"
client = openai.OpenAI()

def retrieve(question, index, corpus, top_k=5):
    q_emb = embedder.encode(
        [question], normalize_embeddings=True, device="cuda"
    ).astype(np.float32)
    _, idxs = index.search(q_emb, top_k)
    return [corpus[i] for i in idxs[0]]

def generate(question, passages, model="gpt-4o-mini"):
    ctx = "\n\n".join(
        f"[{i+1}] {p['title']}: {p['text'][:400]}"
        for i, p in enumerate(passages)
    )
    resp = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content":
            f"Answer in 1-5 words using ONLY the context.\n\n"
            f"Context:\n{ctx}\n\nQuestion: {question}\nAnswer:"}],
        temperature=0, max_tokens=50
    )
    return resp.choices[0].message.content.strip()

with open("data/hotpotqa/dev_1k.json") as f:
    hp_data = json.load(f)

predictions_naive = []
latencies = []

for item in tqdm(hp_data, desc="Naive RAG — HotpotQA"):
    q   = item["question"]
    ans = item["answer"]
    t0  = time.time()
    passages = retrieve(q, index_hp, corpus_hp)
    pred     = generate(q, passages)
    lat      = time.time() - t0
    predictions_naive.append({
        "id":           item.get("id", ""),
        "question":     q,
        "prediction":   pred,
        "gold_answers": [ans],
        "hops":         2,
        "latency":      round(lat, 3),
        "retrieved":    [p["title"] for p in passages]
    })
    latencies.append(lat)

with open("results/naive_rag_hotpotqa.json", "w") as f:
    json.dump(predictions_naive, f, indent=2)

r_naive = evaluate(predictions_naive)
r_naive["avg_latency"] = round(sum(latencies)/len(latencies), 2)
r_naive["system"] = "NaiveRAG-bge-large-gpt4omini"
print(json.dumps(r_naive, indent=2))

with open("results/naive_rag_hotpotqa_scores.json", "w") as f:
    json.dump(r_naive, f, indent=2)

In [ ]:
# ── CELL 7: Install FlashRAG + download corpus ──────────────────────
!git clone https://github.com/RUC-NLPIR/FlashRAG /content/FlashRAG --quiet
!pip install -e /content/FlashRAG --quiet

# Download FlashRAG's pre-built Wikipedia retrieval corpus (stream 500k passages)
import jsonlines
from datasets import load_dataset

os.makedirs("/content/corpus", exist_ok=True)
corpus_path = "/content/corpus/wiki.jsonl"

if not os.path.exists(corpus_path):
    print("Streaming Wikipedia corpus (500k passages)...")
    corpus_ds = load_dataset(
        "RUC-NLPIR/FlashRAG_datasets", "wiki18_100w",
        split="train", streaming=True
    )
    with jsonlines.open(corpus_path, "w") as writer:
        for i, row in enumerate(corpus_ds):
            writer.write(row)
            if i % 50000 == 0: print(f"  {i:,} passages...")
            if i >= 500_000: break
    print(f"Corpus written: {corpus_path}")
else:
    print(f"Corpus already exists: {corpus_path}")

In [ ]:
# ── CELL 7b: Run IRCoT via FlashRAG ─────────────────────────────────
import yaml

config = {
    "data_dir":         "data/",
    "index_path":       "/content/corpus/wiki_index",
    "corpus_path":      "/content/corpus/wiki.jsonl",
    "model2path":       {"e5": "intfloat/e5-base-v2"},
    "retrieval_method": "e5",
    "generator_model":  "gpt-4o-mini",
    "openai_api_key":   os.environ["OPENAI_API_KEY"],
    "retrieval_topk":   5,
    "save_dir":         f"{PROJECT}/results/flashrag/",
    "gpu_id":           "0",
    "dataset_name":     "hotpotqa",
    "split":            "dev",
    "test_sample_num":  1000,
}

with open("/content/flashrag_config.yaml", "w") as f:
    yaml.dump(config, f)

%cd /content/FlashRAG
!python run_exp.py \
    --method ircot \
    --config_path /content/flashrag_config.yaml \
    --save_note ircot_hotpotqa_1k
%cd {PROJECT}

In [ ]:
# ── CELL 8: Compile Phase 1 results table ──────────────────────────
import glob

results_all = {}

with open("results/naive_rag_hotpotqa_scores.json") as f:
    results_all["Naive RAG"] = json.load(f)

for method in ["ircot", "selfrag", "adaptive_rag"]:
    pattern = f"results/flashrag/{method}*scores*.json"
    files = glob.glob(pattern)
    if files:
        with open(sorted(files)[-1]) as f:
            results_all[method.upper()] = json.load(f)

print("\n" + "="*65)
print(f"{'PHASE 1 BASELINE RESULTS':^65}")
print(f"{'HotpotQA fullwiki — 1000 dev samples':^65}")
print("="*65)
print(f"{'System':<22} {'EM':>7} {'F1':>7} {'Latency':>10}")
print("-"*65)
for sys_name, res in results_all.items():
    em  = res.get("EM",  "—")
    f1  = res.get("F1",  "—")
    lat = res.get("avg_latency", "—")
    lat_str = f"{lat}s" if isinstance(lat, (int, float)) else lat
    print(f"{sys_name:<22} {str(em):>7} {str(f1):>7} {lat_str:>10}")
print("="*65)

print("\nLiterature reference ranges:")
ref = {
    "Naive RAG":  ("35-45", "48-58"),
    "Self-RAG":   ("48-55", "60-66"),
    "IRCoT":      ("50-56", "63-68"),
    "HopRAG":     ("52-58", "65-72"),
}
for s, (e, f) in ref.items():
    print(f"  {s:<20} EM: {e:>8}   F1: {f}")
print("\nWithin ±3 pts of range = environment valid → proceed to Phase 2")

with open("results/phase1_summary.json", "w") as f:
    json.dump(results_all, f, indent=2)
print("\nSaved: results/phase1_summary.json")